# Recomendacion basada en perfiles - clase 07
Ya revisada. Son consultas SQL para obtener recomendaciones basadas en preferencias de género y autor (llamadas también perfiles)

In [ ]:
import pandas as pd
import sqlite3

### Claude explanation (not from the professor)
Selects the reader whose profile is built. Swap the commented line to try another reader.

In [ ]:
id_lector = "popocito"
#id_lector = "c3po"

### Claude explanation (not from the professor)
Builds the reader's **genre profile**: the average rating they gave each genre, divided by 10 so it falls in 0–1. The commented `COUNT(*)` line is the alternative: weight genres by how many books were read rather than how well they were rated.

In [ ]:
conn = sqlite3.connect('data.db')

df_perfil_genero = pd.read_sql_query("""
-- SELECT genero, COUNT(*) AS valor
SELECT genero, AVG(rating) / 10 AS valor
  FROM interacciones AS i JOIN libros AS l ON i.id_libro = l.id_libro
  WHERE id_lector = ?
  GROUP BY 1
  ORDER BY 2 DESC
""", conn, params=[id_lector])

print(df_perfil_genero.head(10))

conn.close()

                          genero     valor
0    Fantástica, ciencia ficción  0.800000
1      Clásicos de la literatura  0.783333
2       Literatura contemporánea  0.777778
3  Novela negra, intriga, terror  0.705263
4           Biografías, Memorias  0.700000
5                      Narrativa  0.688571
6          Histórica y aventuras  0.666667
7                         Varios  0.650000
8             Infantil y juvenil  0.650000
9              Ficción literaria  0.638462


### Claude explanation (not from the professor)
Turns the genre profile into a `{genre: score}` dictionary, used later to score candidate books.

In [ ]:
dict_generos = dict(zip(df_perfil_genero["genero"].values.tolist(), df_perfil_genero["valor"].values.tolist()))
dict_generos

{'Fantástica, ciencia ficción': 0.8,
 'Clásicos de la literatura': 0.7833333333333333,
 'Literatura contemporánea': 0.7777777777777778,
 'Novela negra, intriga, terror': 0.7052631578947368,
 'Biografías, Memorias': 0.7,
 'Narrativa': 0.6885714285714286,
 'Histórica y aventuras': 0.6666666666666667,
 'Varios': 0.65,
 'Infantil y juvenil': 0.65,
 'Ficción literaria': 0.6384615384615385,
 'Romántica, erótica': 0.62,
 'Ensayo': 0.6,
 'Cocina': 0.5}

### Claude explanation (not from the professor)
Same idea for **authors**. Only authors the reader rated at least 3 times are kept (`HAVING COUNT(*) >= 3`), so a single book doesn't make a favorite author.

In [ ]:
conn = sqlite3.connect('data.db')

df_perfil_autor = pd.read_sql_query("""
-- SELECT autor, COUNT(*) AS valor
SELECT autor, AVG(rating) / 10 AS valor
  FROM interacciones AS i JOIN libros AS l ON i.id_libro = l.id_libro
  WHERE id_lector = ?
  GROUP BY 1
  HAVING COUNT(*) >= 3
  ORDER BY 2 DESC
""", conn, params=[id_lector])

print(df_perfil_autor.head(10))

conn.close()

                      autor     valor
0               DANON, KATE  0.833333
1   GARCÍA MÁRQUEZ, GABRIEL  0.814286
2        RUIZ ZAFÓN, CARLOS  0.800000
3           HEATH, LORRAINE  0.800000
4           DELIBES, MIGUEL  0.780000
5             BROWN, SANDRA  0.775000
6          REDONDO, DOLORES  0.766667
7             KLEYPAS, LISA  0.766667
8              AUSTEN, JANE  0.750000
9             HOWARD, LINDA  0.733333
10             BALOGH, MARY  0.733333
11          GRAHAM, HEATHER  0.725000
12          ASENSI, MATILDE  0.700000
13          ALLENDE, ISABEL  0.700000
14          CASANOVAS, ANNA  0.690000
15            FOLEY, GAELEN  0.666667
16           GARWOOD, JULIE  0.642857
17         AMARILLO, NOELIA  0.640000
18           LARSSON, STIEG  0.633333
19       BONELLI, FLORENCIA  0.633333
20          HIDALGO, NIEVES  0.620000
21          SICILIA, MARISA  0.566667
22           KINSALE, LAURA  0.500000
23           DEVERAUX, JUDE  0.500000
24            ARDEY, OLIVIA  0.471429
25          

### Claude explanation (not from the professor)
Author profile as an `{author: score}` dictionary.

In [ ]:
dict_autores = dict(zip(df_perfil_autor["autor"].values.tolist(), df_perfil_autor["valor"].values.tolist()))
dict_autores

{'DANON, KATE': 0.8333333333333334,
 'GARCÍA MÁRQUEZ, GABRIEL': 0.8142857142857143,
 'RUIZ ZAFÓN, CARLOS': 0.8,
 'HEATH, LORRAINE': 0.8,
 'DELIBES, MIGUEL': 0.78,
 'BROWN, SANDRA': 0.775,
 'REDONDO, DOLORES': 0.7666666666666667,
 'KLEYPAS, LISA': 0.7666666666666667,
 'AUSTEN, JANE': 0.75,
 'HOWARD, LINDA': 0.7333333333333333,
 'BALOGH, MARY': 0.7333333333333333,
 'GRAHAM, HEATHER': 0.725,
 'ASENSI, MATILDE': 0.7,
 'ALLENDE, ISABEL': 0.7,
 'CASANOVAS, ANNA': 0.6900000000000001,
 'FOLEY, GAELEN': 0.6666666666666667,
 'GARWOOD, JULIE': 0.6428571428571429,
 'AMARILLO, NOELIA': 0.64,
 'LARSSON, STIEG': 0.6333333333333333,
 'BONELLI, FLORENCIA': 0.6333333333333333,
 'HIDALGO, NIEVES': 0.62,
 'SICILIA, MARISA': 0.5666666666666667,
 'KINSALE, LAURA': 0.5,
 'DEVERAUX, JUDE': 0.5,
 'ARDEY, OLIVIA': 0.4714285714285714,
 'QUINN, JULIA': 0.4666666666666667}

### Claude explanation (not from the professor)
Same idea for **publication year**, grouped into decades with `CASE`. The `WHERE` clause drops empty or malformed years.

In [ ]:
conn = sqlite3.connect('data.db')

df_perfil_anio_edicion = pd.read_sql_query("""
-- SELECT anio_edicion, COUNT(*) AS valor
SELECT
    CASE
      WHEN anio_edicion < 1960 THEN 'viejos'
      WHEN anio_edicion >= 1960 AND anio_edicion <= 1969 THEN '60s'
      WHEN anio_edicion >= 1970 AND anio_edicion <= 1979 THEN '70s'
      WHEN anio_edicion >= 1980 AND anio_edicion <= 1989 THEN '80s'
      WHEN anio_edicion >= 1990 AND anio_edicion <= 1999 THEN '90s'
      WHEN anio_edicion >= 2000 AND anio_edicion <= 2009 THEN '00s'
      WHEN anio_edicion >= 2010 AND anio_edicion <= 2019 THEN '10s'
      WHEN anio_edicion >= 2020 AND anio_edicion <= 2029 THEN '20s'
      ELSE 'futuros'
    END AS anio_edicion,
AVG(rating) / 10 AS valor
  FROM interacciones AS i JOIN libros AS l ON i.id_libro = l.id_libro
  WHERE id_lector = ? AND anio_edicion IS NOT NULL AND anio_edicion != ' ' AND anio_edicion != ' (200'
  GROUP BY 1
  ORDER BY 2 DESC
""", conn, params=[id_lector])

print(df_perfil_anio_edicion.head(10))

conn.close()

  anio_edicion     valor
0          00s  0.678571
1          90s  0.650000
2          10s  0.632536
3          20s  0.610000


### Claude explanation (not from the professor)
Decade profile as a `{decade: score}` dictionary.

In [ ]:
dict_anio_edicion = dict(zip(df_perfil_anio_edicion["anio_edicion"].values.tolist(), df_perfil_anio_edicion["valor"].values.tolist()))
dict_anio_edicion

{'00s': 0.6785714285714286,
 '90s': 0.65,
 '10s': 0.6325358851674642,
 '20s': 0.61}

### Claude explanation (not from the professor)
Gets the **candidate books**: books the reader hasn't rated yet.

**Note:** the query has two `?` placeholders, but `params` passes only `id_lector`, so SQLite raises an "Incorrect number of bindings" error. The `genero IN (?)` filter needs its own value, for example the reader's top genres.

In [ ]:
conn = sqlite3.connect('data.db')

df = pd.read_sql_query("""
SELECT id_libro, genero, autor, anio_edicion
  FROM libros AS l
  WHERE id_libro NOT IN (SELECT id_libro FROM interacciones WHERE id_lector = ?)
    AND genero IN (?)
""", conn, params=[id_lector])

print(df.head(10))

conn.close()

                                            id_libro  \
0                         las-particulas-elementales   
1                              quien-domina-el-mundo   
2                            antimanual-de-filosofia   
3  momentos-estelares-de-la-humanidad-catorce-min...   
4                            la-invencion-del-pasado   
5                                         serotonina   
6                                la-edad-de-la-ira-1   
7                                     en-el-enjambre   
8                                      luz-de-agosto   
9               el-cuerpo-humano-guia-para-ocupantes   

                     genero                autor anio_edicion  
0                 Narrativa  HOUELLEBECQ, MICHEL         2006  
1                    Ensayo        CHOMSKY, NOAM         2016  
2                    Ensayo       ONFRAY, MICHEL         2011  
3     Histórica y aventuras        ZWEIG, STEFAN         2002  
4  Lecturas complementarias  MURADO, MIGUEL-ANXO         2013  

### Claude explanation (not from the professor)
Scores each candidate by adding its genre, author and decade scores from the profiles (0 when the value isn't in the profile), then keeps the top 20. As the `FALTA PONDERAR` comment says, the three parts aren't weighted yet.

**Note:** `anio_edicion` here is the raw year (e.g. `2005`), but the decade profile's keys are labels such as `'00s'`, so `anio_edicion_score` is always 0. The year needs the same decade grouping first.

In [ ]:
df["genero_score"] = df["genero"].apply(lambda x: dict_generos.get(x, 0))
df["autor_score"] = df["autor"].apply(lambda x: dict_autores.get(x, 0))
df["anio_edicion_score"] = df["anio_edicion"].apply(lambda x: dict_anio_edicion.get(x, 0))
df["score"] = df["genero_score"] + df["autor_score"] + df["anio_edicion_score"] # FALTA PONDERAR
df = df.sort_values(by="score", ascending=False).head(20)
df

,id_libro,genero,autor,anio_edicion,genero_score,autor_score,anio_edicion_score,score
2676,vivir-para-contarla,Literatura contemporánea,"GARCÍA MÁRQUEZ, GABRIEL",2003,0.777778,0.814286,0,1.592063
15249,diatriba-de-amor-contra-un-hombre-sentado,Literatura contemporánea,"GARCÍA MÁRQUEZ, GABRIEL",2004,0.777778,0.814286,0,1.592063
139,cien-anos-de-soledad-1,Literatura contemporánea,"GARCÍA MÁRQUEZ, GABRIEL",2017,0.777778,0.814286,0,1.592063
2376,senora-de-rojo-sobre-fondo-gris,Literatura contemporánea,"DELIBES, MIGUEL",2009,0.777778,0.780000,0,1.557778
31480,los-santos-inocentes-1,Literatura contemporánea,"DELIBES, MIGUEL",2019,0.777778,0.780000,0,1.557778
3924,la-hoja-roja,Literatura contemporánea,"DELIBES, MIGUEL",2010,0.777778,0.780000,0,1.557778
13442,cartas-de-amor-de-un-sexagenario-voluptuoso,Literatura contemporánea,"DELIBES, MIGUEL",2010,0.777778,0.780000,0,1.557778
13450,diario-de-un-inmigrante,Literatura contemporánea,"DELIBES, MIGUEL",2008,0.777778,0.780000,0,1.557778
18272,aun-es-de-dia,Literatura contemporánea,"DELIBES, MIGUEL",2010,0.777778,0.780000,0,1.557778
31416,con-la-escopeta-al-hombro,Literatura contemporánea,"DELIBES, MIGUEL",1980,0.777778,0.780000,0,1.557778
